**// IMPORTS**

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight


**// TASK 2.1**

In [2]:
pairs = pd.read_csv("../task_01/final_before_modelling.csv")  # columns: dblp_id, scholar_id
gt = pd.read_csv("../data/DBLP-Scholar_perfectMapping.csv")  # columns: dblp_id, scholar_id

pairs["dblp_year"] = pd.to_numeric(pairs["dblp_year"]).astype("Int64")
pairs["sch_year"]  = pd.to_numeric(pairs["sch_year"]).astype("Int64")

# create a key to join on
gt["pair_key"] = gt["idDBLP"].astype(str) + "##" + gt["idScholar"].astype(str)
pairs["pair_key"] = pairs["dblp_id"].astype(str) + "##" + pairs["sch_id"].astype(str)

gt_keys = set(gt["pair_key"])

# label: 1 if in ground truth, else 0
pairs["label"] = pairs["pair_key"].isin(gt_keys).astype(int)

print(pairs["label"].value_counts())

label
0    29782
1     4732
Name: count, dtype: int64


**// TASK 2.2**

In [3]:
# JACCARD VENUES
def jaccard_venue_from_str(a, b):
    A = set(str(a).split())
    B = set(str(b).split())
    if not A and not B:
        return 0.0
    return len(A & B) / len(A | B)

pairs["jaccard_venue"] = pairs.apply(
    lambda r: jaccard_venue_from_str(r["dblp_venue"], r["sch_venue"]),
    axis=1
)


In [4]:
# COS SIM AUTHORS
def authors_to_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    return str(x)

dblp_auth_text = pairs["dblp_authors"].apply(authors_to_text)
sch_auth_text  = pairs["sch_authors"].apply(authors_to_text)

all_auth = pd.concat([dblp_auth_text, sch_auth_text])

vec_auth = TfidfVectorizer().fit(all_auth)

A_auth = vec_auth.transform(dblp_auth_text)
B_auth = vec_auth.transform(sch_auth_text)

# 1D array: cosine similarity per row
cos_dist = paired_cosine_distances(A_auth, B_auth)  # distance in [0,2]
pairs["cos_authors"] = 1 - cos_dist          

In [5]:
# SHARED TOKENS TITLE
def title_tokens(x):
    if isinstance(x, list):
        return [str(t).lower() for t in x]
    return str(x).lower().split()

def shared_token_count(a, b):
    A = set(title_tokens(a))
    B = set(title_tokens(b))
    return len(A & B)

pairs["title_shared_tokens"] = pairs.apply(
    lambda r: shared_token_count(r["dblp_title"], r["sch_title"]),
    axis=1
)

In [6]:
def title_len(x):
    if isinstance(x, list):
        return len(x)
    return len(str(x).split())

pairs["title_len_diff"] = pairs.apply(
    lambda r: abs(title_len(r["dblp_title"]) - title_len(r["sch_title"])),
    axis=1
)


In [7]:
pairs.head()

,i_dblp,j_sch,dblp_id,sch_id,cos_title,jaccard_title,dice_title,Levenshtein_title,year_score,dice_authors,...,dblp_venue,sch_venue,dblp_year,sch_year,pair_key,label,jaccard_venue,cos_authors,title_shared_tokens,title_len_diff
0,1845,55524,conf/vldb/ChakrabartiDAR97,AyDb4G2EiGwJ,0.038753,0.166667,0.285714,0.304348,0.8,1.000000,...,['vldb'],"['vldb', 'journal', 'international', 'journal'...",1997,1998,conf/vldb/ChakrabartiDAR97##AyDb4G2EiGwJ,0,0.0,1.000000,2,1
1,2174,6007,conf/vldb/HammelP02,SU2XS1MVb40J,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,...,['vldb'],['vldb'],2002,2002,conf/vldb/HammelP02##SU2XS1MVb40J,1,1.0,1.000000,8,0
2,1045,10586,conf/vldb/GravanoIJKMS01,w0kVbjS1psIJ,0.005072,0.052632,0.100000,0.292683,0.0,0.444444,...,['vldb'],"['proceedings', '26th', 'intâ', 'l', 'conferen...",2001,<NA>,conf/vldb/GravanoIJKMS01##w0kVbjS1psIJ,0,0.0,0.443811,1,2
3,243,20286,conf/sigmod/GuoSBS03,ho3IAnDHcrcJ,0.064790,0.153846,0.266667,0.130435,0.8,0.571429,...,"['sigmod', 'conference']","['proceedings', '2004', 'acm', 'sigmod', 'inte...",2003,2004,conf/sigmod/GuoSBS03##ho3IAnDHcrcJ,0,0.0,0.576489,2,1
4,1986,8400,conf/vldb/WangZL03,SAbNG8szVhQJ,0.000000,0.000000,0.000000,0.240000,0.8,0.333333,...,['vldb'],[],2003,2002,conf/vldb/WangZL03##SAbNG8szVhQJ,0,0.0,0.100772,0,3


**// PREPROCESS TRAINING DATA**

Handle missing values, drop duplicates, and scale numeric features for model-ready input.

In [ ]:
numeric_features = [
    'cos_title',
    'jaccard_title',
    'dice_title',
    'Levenshtein_title',
    'year_score',
    'dice_authors',
    'jaccard_venue',
    'cos_authors',
    'title_shared_tokens',
    'title_len_diff',
]

before = len(pairs)
pairs = pairs.drop_duplicates(subset='pair_key').reset_index(drop=True)
print(f'Removed {before - len(pairs)} duplicate rows')

pairs[numeric_features] = pairs[numeric_features].fillna(0)

scaler = StandardScaler()
scaled_array = scaler.fit_transform(pairs[numeric_features])

scaled_feature_df = pd.DataFrame(
    scaled_array,
    columns=[f'{c}_scaled' for c in numeric_features],
)

train_df = pd.concat(
    [pairs[['pair_key', 'label']].reset_index(drop=True), scaled_feature_df],
    axis=1,
)

train_df.head()


Address class imbalance with class weights (pass `class_weight` to estimators).

In [ ]:
class_counts = train_df['label'].value_counts()
classes = np.array(sorted(class_counts.index))
class_weights = compute_class_weight(
    class_weight='balanced', classes=classes, y=train_df['label']
)
class_weight_dict = {cls: weight for cls, weight in zip(classes, class_weights)}

print('Class distribution:', class_counts.to_dict())
print('Class weights:', class_weight_dict)
